# What do Embeddings actually mean?


In [1]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [2]:
sentence1="The cat sat on the mat"
sentence2="A kitten rested on the rug."
sentence3="The stock market crashed today."

In [3]:
embedding1 = model.encode(sentence1)
embedding2 = model.encode(sentence2)
embedding3 = model.encode(sentence3)


In [4]:
embedding1.shape  #1D array with 384 elements

(384,)

**TASK** - compute the cosine similarity between 1&2 and between 1&3.

In [5]:
import numpy as np

In [6]:
cosine1 = np.dot(embedding1,embedding2)/(np.linalg.norm(embedding1)*np.linalg.norm(embedding2))

In [7]:
def cosine(e1,e2):
  return np.dot(e1,e2)/(np.linalg.norm(e1)*np.linalg.norm(e2))

In [8]:
cosine1

np.float32(0.61364627)

In [9]:
cosine2 = np.dot(embedding1,embedding3)/(np.linalg.norm(embedding1)*np.linalg.norm(embedding3))
cosine2

np.float32(0.09248571)

In [10]:
cosine1-cosine2

np.float32(0.52116054)

This makes sense as A and B talks about similar patterns while C is a totally different sentence.

# Breaking down a para - Beginning to learn chunking

In [11]:
text= """Artificial intelligence is transforming healthcare.
Doctors now use AI to detect cancer early from scans.
Meanwhile, AI is also changing finance.
Banks use machine learning to detect fraud in real time.
Education is another field being disrupted.
Personalized AI tutors can adapt to each student's learning pace."""

In [12]:
text

"Artificial intelligence is transforming healthcare. \nDoctors now use AI to detect cancer early from scans. \nMeanwhile, AI is also changing finance. \nBanks use machine learning to detect fraud in real time. \nEducation is another field being disrupted. \nPersonalized AI tutors can adapt to each student's learning pace."

In [13]:
def chunk_by_sentences(text,chunk_size=2):
  temp=text.split(". ") # temp holds one sen each
  chunks=[]
  for i in range(0,len(temp),chunk_size):
    chunks.append(" ".join(temp[i:i+chunk_size]))

  return chunks

In [14]:
chunks = chunk_by_sentences(text)

In [15]:
chunks

['Artificial intelligence is transforming healthcare \nDoctors now use AI to detect cancer early from scans',
 '\nMeanwhile, AI is also changing finance \nBanks use machine learning to detect fraud in real time',
 "\nEducation is another field being disrupted \nPersonalized AI tutors can adapt to each student's learning pace."]

In [16]:
for i in range(0,len(chunks)):
  print(f"Chunk {i} is {chunks[i]}")

Chunk 0 is Artificial intelligence is transforming healthcare 
Doctors now use AI to detect cancer early from scans
Chunk 1 is 
Meanwhile, AI is also changing finance 
Banks use machine learning to detect fraud in real time
Chunk 2 is 
Education is another field being disrupted 
Personalized AI tutors can adapt to each student's learning pace.


# Chunks + embedding

In [17]:
chunks_info=[]
embeddings = model.encode(chunks)
embeddings

array([[ 0.01650238,  0.05942089, -0.00032181, ..., -0.0163377 ,
         0.10590293, -0.03322232],
       [-0.03081957, -0.06293319, -0.04523003, ..., -0.02323625,
         0.07001409, -0.04001866],
       [-0.02954879, -0.00256975,  0.01228646, ...,  0.03035541,
        -0.04795494,  0.04309963]], dtype=float32)

In [18]:
for i in range(0,len(chunks)):
  entry={
      "id":i,
      "text":chunks[i],
      "embedding":embeddings[i]
  }
  chunks_info.append(entry)


In [19]:
for entry in chunks_info:
    print(f"Chunk {entry['id']}: {entry['text'][:50]}...")
    print(f"Embedding preview: {entry['embedding'][:3]}")
    print()

Chunk 0: Artificial intelligence is transforming healthcare...
Embedding preview: [ 0.01650238  0.05942089 -0.00032181]

Chunk 1: 
Meanwhile, AI is also changing finance 
Banks use...
Embedding preview: [-0.03081957 -0.06293319 -0.04523003]

Chunk 2: 
Education is another field being disrupted 
Perso...
Embedding preview: [-0.02954879 -0.00256975  0.01228646]



# Retrieval

In [20]:
def retrieve(query,chunks_info,top_n=1):
  query_embedding = model.encode(query)
  cos=[]
  for entry in chunks_info:
    score=cosine(query_embedding,entry['embedding'])
    cos.append((score, entry))

  # to retrive top chunks we need to sort cos, 0-th is score in the tuple
  cos.sort(key=lambda x: x[0], reverse=True)

  return [entry['text'] for score,entry in cos[:top_n]]




In [21]:
query=["Which fields are being changed by AI?","How is AI used in medicine?"]

In [22]:
for q in query:
  print(f"Query:  {q}")

  print(f"Best match : {retrieve(q,chunks_info)}")


Query:  Which fields are being changed by AI?
Best match : ['\nMeanwhile, AI is also changing finance \nBanks use machine learning to detect fraud in real time']
Query:  How is AI used in medicine?
Best match : ['Artificial intelligence is transforming healthcare \nDoctors now use AI to detect cancer early from scans']


# RAG Pipeline

In [23]:
!pip -q install groq

In [24]:
from groq import Groq
client=Groq(api_key="************************")
print("Connected!")

Connected!


In [25]:
response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
      {
        "role": "user",
        "content": "What is the formula of Caustic Soda?"
      }
    ]
)

In [26]:
response.choices[0].message.content

'The formula for Caustic Soda is NaOH, which stands for Sodium Hydroxide.'

In [27]:
prompt="""
You are a helpful assistant. Answer the question using ONLY the context below.
If the answer is not in the context, say "I don't know".
Context: {context}
Question: {query}
"""

In [28]:
def rag_pipeline(query):
  context = retrieve(query,chunks_info)
  response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
      {
        "role": "user",
        "content": f"""
            You are a helpful assistant. Answer the question using ONLY the context below.
            If the answer is not in the context, say "I don't know".
            Context: {context}
            Question: {query}
     """
      }
    ]

)
  return response.choices[0].message.content



In [29]:
print(rag_pipeline("How does AI help doctors?"))
print(rag_pipeline("Is AI used in space exploration?"))

AI helps doctors detect cancer early from scans.
I don't know.


What if I have 1000 or more chunks, it will be quite difficult to manage it!

# Vector DB - Chroma

ChromaDB is going to replace!

- No more chunks_info list
- No more manual cosine similarity loop

In [ ]:
#!pip install -q chromadb opentelemetry-sdk --upgrade  --> run this and restart

In [30]:
import chromadb

In [31]:
client_db = chromadb.Client()
collection= client_db.create_collection(name="my_rag")

In [36]:
ids=[]
documents=[]
embeddings=[]
for entry in chunks_info:
  ids.append(str(entry['id'])) # chromadb needs ids as strings
  documents.append(entry['text'])
  embeddings.append(entry['embedding'])

In [37]:
collection.add(
    ids=ids,
    documents=documents,
    embeddings=embeddings,

)

In [38]:
print(collection.count())

3


ChromaDB has a built in query method. In plain English it does this:

- query embedding
- It finds the most similar chunks internally
- It returns the results

In [39]:
query_embed = model.encode("How does AI help doctors?")
results = collection.query(query_embeddings=[query_embed],n_results=2)
print(results)

{'ids': [['0', '2']], 'embeddings': None, 'documents': [['Artificial intelligence is transforming healthcare \nDoctors now use AI to detect cancer early from scans', "\nEducation is another field being disrupted \nPersonalized AI tutors can adapt to each student's learning pace."]], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[None, None]], 'distances': [[0.7146301865577698, 1.0367941856384277]]}


Decoding
- id 0 and 2 are the top 2 relevant chunks
- distances show how different each chunk is from query. **lower means MORE SIMILAR**
- ChromaDB doesn't return embeddings by default to save memory

In [40]:
def rag_pipeline_2POINT0(query):
  query_embed = model.encode(query)
  res = collection.query(query_embeddings=[query_embed],n_results=1)
  context=res['documents']
  response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
      {
        "role": "user",
        "content": f"""
            You are a helpful assistant. Answer the question using ONLY the context below.
            If the answer is not in the context, say "I don't know".
            Context: {context}
            Question: {query}
     """
      }
    ]

)
  return response.choices[0].message.content

In [41]:
print(rag_pipeline_2POINT0("How does AI help doctors?"))
print(rag_pipeline_2POINT0("Is AI used in space exploration?"))

AI helps doctors detect cancer early from scans.
I don't know.
